# 16. Web Search Tool

Lesson notes: [SS12 · The Web Search Tool](../s06_tool_use_with_claude/ss12_the_web_search_tool/index.md)

This notebook demonstrates how to enable Claude's built-in web search tool, send a prompt, and inspect the returned response blocks and citations.

## 1) Setup Client and Model

This cell initializes the Anthropic client and selects the model used for web-search-enabled requests.

Reference notes: [Open SS12 Notes](../s06_tool_use_with_claude/ss12_the_web_search_tool/index.md)

What this does:
- Loads environment variables from your local setup.
- Creates a reusable API client.
- Sets the target model used for all calls in this notebook.

In [14]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-4-5"

## 2) Message and Chat Helpers

This cell defines helper functions for creating conversation messages and sending API requests.

What this does:
- Normalizes user and assistant messages into the expected API structure.
- Wraps request parameters in a reusable `chat(...)` function.
- Extracts text blocks from Claude responses for easier display and debugging.

In [15]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## 3) Prompt Claude with Web Search Enabled

This cell prepares a prompt and sends it to Claude.

What this does:
- Starts a fresh message list.
- Adds the user query that should trigger web search behavior.
- Sends the request and prints the structured response object so you can inspect tool-use and citation blocks.

In [ ]:
web_search_schema = {
    "type" : "web_search_20250305",
    "name" : "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [18]:
messages = []
add_user_message(
    messages,
    """
    what are the best leg exercises ? 
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_011CeuprjnBXv84jPAx7sxQR', container=None, content=[ServerToolUseBlock(id='srvtoolu_01HYswSbv86HkmULCbpXR7KE', caller=None, input={'query': 'Nestle news latest 2026'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EuAQCioIExgCIiQyZmU2YTU4NC1jMzViLTQ5NTMtYTMwOC03OTU2Yzk5M2U3ODMSDG2Et20+Yc9EfEbhZxoMNiJ5PC3RqI1T7PnLIjDO4GHmz3Hkfyo0Eo1Z6+F6lzDIV+jzYhkXSlXwYBs5rc0B+yJwJddK3Ru6BaZKJu8q4w90/DSAarMd0zTOzKoZsqImuXd9/PSUd2A9OuVhsqlhaSnixPTs8H1TAWdjGeApyhHVZ/kkzb5t7GIwSs/nFKw1JYr00/d58oLJahVbkT2qdqQblMAVJl9GoK5+mpnOzzkObjO7NBZLIPfTnNyHFFgnsewdac69FD4OOUGiWclXowIYHi15fXgD3S1ArEkMApfwW0gbecpvIYQShZ2lLQv2G3ds6ted4DV5dMz7uWKGkEVpa8nzFk5/Q3icAyoOjFHbXKhP8lnJTkFcX3RHqRCUcvIsRYJp7j5w34M63+IiAh/bpYfn40OQXQ9QZ6wrgf7qc60XXWZogN8ww0QzqRnobMkFnbOj4shWR5twhiUWt7whHiN9jjhBDMTq8zt3VYiWNYtZZ9kldwfKgPDBQTVJcCgcw3SmioAPwTTh2C2hXKgz2mYatLKG5ibXPCk2CbY6QJ7NvPjCLtdnoq9uzULL095E6CcrjkIEd7NvCfY5iM